# Final Model Training

## Objective

Train and select the final True-Gate model **from B01–B07**, using the conclusions established by the completed research questions.

This notebook does **not** load a historical MAIN v7/v8 model.

It performs:

1. seven-fold password-level model-family comparison;
2. detailed model analysis;
3. deterministic model-family selection;
4. retraining of the selected family on all B01–B07;
5. final model freezing.

A new password collected **after the final freeze** is required for the final unbiased test.

---

## Fixed experimental design from the RQs

Only the **model family** changes.

- Task: controlled **10-candidate ranking**
- Signal: **Current movement only**
- Information content: **Dual-full 695-D representation**
- Wheel handling: **wheel-specific**
- Direction handling: **CW and CCW are separate physical domains**
- Repeats: **A+B used for training**
- Primary inference: matched A/B score fusion
- Evaluation: leave one complete password batch out
- No password/profile/repeat/ordinal/digit/identifier shortcut inputs

### A/B fusion

Each model returns one scalar candidate score.

Within each 10-candidate A or B scan, scores are standardised. Matching physical-digit scores are then averaged across A/B and the ten fused candidates are ranked.

CW and CCW remain separate decisions.

---

# Model comparison

## Track A — engineered-feature models

All models receive the flat 695-D vector.

1. Logistic Regression
2. RBF-SVM
3. HistGradientBoosting
4. MLP

## Track B — structured deep models

The **same 695 values** are reorganised without adding or removing information:

### Temporal tensor

The three log-Mel blocks are each `6 × 32`:

- Ch1 log-Mel: `6 × 32`
- Ch2 log-Mel: `6 × 32`
- Ch1 − Ch2 log-Mel: `6 × 32`

They are concatenated at each temporal bin:

`6 time bins × 96 acoustic features`

### Global vector

The remaining features are:

- Ch1 spectral + envelope = 58
- Ch2 spectral + envelope = 58
- cross-channel scalars = 3

Total global vector:

`119 features`

Thus every deep model still uses all **695 values exactly once**:

`6 × 96 + 119 = 695`

### Deep model families

All sequence models share the **same convolutional front-end and the same global-feature branch**. Only the temporal modelling block changes:

1. CNN
2. CNN + LSTM
3. CNN + BiLSTM
4. CNN + BiGRU
5. CNN + Transformer

This follows a controlled model-comparison design: common input, common preprocessing, common convolutional feature extraction and common output head; only the temporal mechanism changes.

---

# Evaluation framework

The physical task is ranking, therefore the **primary model-selection metric is A/B-fused password-LOPO Top-1**.

The complete report also saves:

- Top-1 / Top-2 / Top-3
- mean true rank
- MRR
- Accuracy
- Macro-F1
- Weighted-F1
- classification report
- 10 × 10 confusion matrix
- per-digit precision / recall / F1 / support
- per-wheel performance
- per-direction performance
- per-password performance
- training time
- model size
- neural-model train / validation loss curves
- neural-model validation Top-1 curves
- early-stopping epoch
- password-level paired model comparisons

For neural models, an **inner password-grouped validation** is used only for early stopping:

- outer fold: 1 password held out for evaluation;
- among the remaining 6 passwords: 1 deterministic password is used for validation;
- after choosing the best epoch, the model is retrained from scratch on all 6 outer-training passwords for exactly that many epochs;
- only then is the outer held-out password scored.

This prevents the outer password from influencing training duration.


## 1. Environment and reproducibility

In [1]:
from google.colab import drive
from pathlib import Path
import io
import json
import math
import os
import pickle
import random
import shutil
import time
import hashlib
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

drive.mount(
    "/content/drive"
)

SEED = 20260820

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )
except Exception:
    pass

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "Padlock_Reproduction_v1"
)

results_candidates = [
    PROJECT_ROOT / "results",
    PROJECT_ROOT
    / "Padlock_Reproduction_v1"
    / "results",
]

RESULTS_ROOT = next(
    (
        p
        for p in results_candidates
        if (
            p
            / "07_RQ7_Absolute_vs_Relative"
            / "RQ7_candidate_manifest.csv"
        ).exists()
    ),
    None,
)

if RESULTS_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the active Padlock_Reproduction_v1/results."
    )

MANIFEST_PATH = (
    RESULTS_ROOT
    / "07_RQ7_Absolute_vs_Relative"
    / "RQ7_candidate_manifest.csv"
)

CACHE_DIR = (
    RESULTS_ROOT
    / "feature_cache"
)

RESULT_DIR = (
    RESULTS_ROOT
    / "Final_Model_Training"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FROZEN_DIR = (
    RESULT_DIR
    / "Final_TrueGate_Model"
)

BATCHES = [
    "B01_2085",
    "B02_4369",
    "B03_5720",
    "B04_6893",
    "B05_7246",
    "B06_8451",
    "B07_9638",
]

WHEELS = (
    1,
    2,
    4,
)

DIRECTIONS = (
    "CCW",
    "CW",
)

FEATURE_FAMILIES = [
    "Logistic Regression",
    "RBF-SVM",
    "HistGradientBoosting",
    "MLP",
]

DEEP_FAMILIES = [
    "CNN",
    "CNN + LSTM",
    "CNN + BiLSTM",
    "CNN + BiGRU",
    "CNN + Transformer",
]

MODEL_FAMILIES = (
    FEATURE_FAMILIES
    + DEEP_FAMILIES
)

print(
    "Results root:",
    RESULTS_ROOT,
)

print(
    "Device:",
    DEVICE,
)

print(
    "Model families:",
    len(
        MODEL_FAMILIES
    ),
)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Padlock_Reproduction_v1/results
Device: cuda
Model families: 9


## 2. Load and validate B01–B07

In [2]:
manifest = pd.read_csv(
    MANIFEST_PATH
)

if "row_id" not in manifest.columns:
    manifest[
        "row_id"
    ] = np.arange(
        len(
            manifest
        ),
        dtype=int,
    )

assert len(manifest) == 8400
assert manifest["decision_uid"].nunique() == 840
assert manifest["profile_id"].nunique() == 210
assert manifest.groupby("decision_uid").size().eq(10).all()
assert manifest.groupby("decision_uid")["y"].sum().eq(1).all()

X = np.empty(
    (
        len(
            manifest
        ),
        695,
    ),
    dtype=np.float32,
)

for batch in BATCHES:
    cache_path = (
        CACHE_DIR
        / f"{batch}_695d.npz"
    )

    if not cache_path.exists():
        raise FileNotFoundError(
            cache_path
        )

    cache = np.load(
        cache_path,
        allow_pickle=False,
    )

    run_ids = cache[
        "run_ids"
    ].astype(
        str
    )

    features = cache[
        "features"
    ].astype(
        np.float32
    )

    assert features.shape == (
        1200,
        695,
    )

    index = {
        run_id: i
        for i, run_id
        in enumerate(
            run_ids
        )
    }

    batch_manifest = manifest[
        manifest[
            "batch"
        ]
        == batch
    ]

    rows = batch_manifest[
        "row_id"
    ].to_numpy(
        dtype=int
    )

    X[
        rows
    ] = np.vstack([
        features[
            index[
                run_id
            ]
        ]
        for run_id
        in batch_manifest[
            "run_id"
        ].astype(
            str
        )
    ])

assert X.shape == (
    8400,
    695,
)

assert np.isfinite(
    X
).all()

dataset_summary = pd.DataFrame([
    {
        "password_batches": manifest[
            "batch"
        ].nunique(),
        "profiles": manifest[
            "profile_id"
        ].nunique(),
        "decisions": manifest[
            "decision_uid"
        ].nunique(),
        "candidate_runs": len(
            manifest
        ),
        "feature_count": X.shape[
            1
        ],
    }
])

display(
    dataset_summary
)

,password_batches,profiles,decisions,candidate_runs,feature_count
0,7,210,840,8400,695


## 3. Structured representation for CNN/RNN/Transformer models

In [3]:
def split_structured_features(
    X_flat,
):
    """
    Reorganise the same 695 features without changing information content.

    Flat layout:
      Ch1 log-Mel       0:192       = 6 x 32
      Ch1 global      192:250       = 58
      Ch2 log-Mel     250:442       = 6 x 32
      Ch2 global      442:500       = 58
      Ch1-Ch2 logMel  500:692       = 6 x 32
      cross           692:695       = 3

    Returns:
      sequence: N x 6 x 96
      global:   N x 119
    """

    X_flat = np.asarray(
        X_flat,
        dtype=np.float32,
    )

    ch1_mel = X_flat[
        :,
        0:192,
    ].reshape(
        -1,
        6,
        32,
    )

    ch1_global = X_flat[
        :,
        192:250,
    ]

    ch2_mel = X_flat[
        :,
        250:442,
    ].reshape(
        -1,
        6,
        32,
    )

    ch2_global = X_flat[
        :,
        442:500,
    ]

    diff_mel = X_flat[
        :,
        500:692,
    ].reshape(
        -1,
        6,
        32,
    )

    cross = X_flat[
        :,
        692:695,
    ]

    sequence = np.concatenate(
        [
            ch1_mel,
            ch2_mel,
            diff_mel,
        ],
        axis=2,
    )

    global_features = np.concatenate(
        [
            ch1_global,
            ch2_global,
            cross,
        ],
        axis=1,
    )

    assert sequence.shape[
        1:
    ] == (
        6,
        96,
    )

    assert global_features.shape[
        1
    ] == 119

    assert (
        sequence.shape[
            1
        ]
        * sequence.shape[
            2
        ]
        + global_features.shape[
            1
        ]
        == 695
    )

    return (
        sequence.astype(
            np.float32
        ),
        global_features.astype(
            np.float32
        ),
    )


X_seq, X_global = split_structured_features(
    X
)

print(
    "Structured sequence:",
    X_seq.shape,
)

print(
    "Global features:",
    X_global.shape,
)

Structured sequence: (8400, 6, 96)
Global features: (8400, 119)


## 4. Fixed model configurations

In [4]:
MODEL_CONFIG = {
    "Logistic Regression": {
        "C": 0.10,
        "solver": "liblinear",
    },
    "RBF-SVM": {
        "C": 1.0,
        "gamma": "scale",
    },
    "HistGradientBoosting": {
        "learning_rate": 0.05,
        "max_iter": 160,
        "max_leaf_nodes": 31,
        "l2_regularization": 1.0,
    },
    "MLP": {
        "hidden_1": 256,
        "hidden_2": 128,
        "dropout": 0.30,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "max_epochs": 40,
        "patience": 6,
        "batch_size": 128,
    },
    "deep_common": {
        "conv_channels": 128,
        "global_hidden": 64,
        "head_hidden": 128,
        "dropout": 0.30,
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
        "max_epochs": 35,
        "patience": 6,
        "batch_size": 128,
    },
    "CNN + LSTM": {
        "hidden": 128,
    },
    "CNN + BiLSTM": {
        "hidden_per_direction": 64,
    },
    "CNN + BiGRU": {
        "hidden_per_direction": 64,
    },
    "CNN + Transformer": {
        "heads": 4,
        "feedforward": 256,
        "layers": 1,
    },
}

with open(
    RESULT_DIR
    / "Final_Model_Config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        MODEL_CONFIG,
        f,
        indent=2,
    )

print(
    json.dumps(
        MODEL_CONFIG,
        indent=2,
    )
)

{
  "Logistic Regression": {
    "C": 0.1,
    "solver": "liblinear"
  },
  "RBF-SVM": {
    "C": 1.0,
    "gamma": "scale"
  },
  "HistGradientBoosting": {
    "learning_rate": 0.05,
    "max_iter": 160,
    "max_leaf_nodes": 31,
    "l2_regularization": 1.0
  },
  "MLP": {
    "hidden_1": 256,
    "hidden_2": 128,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "max_epochs": 40,
    "patience": 6,
    "batch_size": 128
  },
  "deep_common": {
    "conv_channels": 128,
    "global_hidden": 64,
    "head_hidden": 128,
    "dropout": 0.3,
    "learning_rate": 0.0005,
    "weight_decay": 0.0001,
    "max_epochs": 35,
    "patience": 6,
    "batch_size": 128
  },
  "CNN + LSTM": {
    "hidden": 128
  },
  "CNN + BiLSTM": {
    "hidden_per_direction": 64
  },
  "CNN + BiGRU": {
    "hidden_per_direction": 64
  },
  "CNN + Transformer": {
    "heads": 4,
    "feedforward": 256,
    "layers": 1
  }
}


## 5. Classical model wrappers

In [5]:
class ClassicalScoreModel:
    def __init__(
        self,
        family,
        seed,
    ):
        self.family = family
        self.seed = int(
            seed
        )
        self.scaler = StandardScaler()
        self.estimator = None

    def fit(
        self,
        X_train,
        y_train,
    ):
        Xs = self.scaler.fit_transform(
            X_train
        )

        if self.family == "Logistic Regression":
            cfg = MODEL_CONFIG[
                self.family
            ]

            self.estimator = LogisticRegression(
                C=cfg[
                    "C"
                ],
                solver=cfg[
                    "solver"
                ],
                class_weight="balanced",
                max_iter=4000,
                random_state=self.seed,
            )

            self.estimator.fit(
                Xs,
                y_train,
            )

        elif self.family == "RBF-SVM":
            cfg = MODEL_CONFIG[
                self.family
            ]

            self.estimator = SVC(
                C=cfg[
                    "C"
                ],
                gamma=cfg[
                    "gamma"
                ],
                kernel="rbf",
                class_weight="balanced",
                probability=False,
                random_state=self.seed,
            )

            self.estimator.fit(
                Xs,
                y_train,
            )

        elif self.family == "HistGradientBoosting":
            cfg = MODEL_CONFIG[
                self.family
            ]

            self.estimator = HistGradientBoostingClassifier(
                learning_rate=cfg[
                    "learning_rate"
                ],
                max_iter=cfg[
                    "max_iter"
                ],
                max_leaf_nodes=cfg[
                    "max_leaf_nodes"
                ],
                l2_regularization=cfg[
                    "l2_regularization"
                ],
                random_state=self.seed,
            )

            sample_weight = compute_sample_weight(
                class_weight="balanced",
                y=y_train,
            )

            self.estimator.fit(
                Xs,
                y_train,
                sample_weight=sample_weight,
            )

        else:
            raise ValueError(
                self.family
            )

        return self

    def score(
        self,
        X_test,
    ):
        Xs = self.scaler.transform(
            X_test
        )

        if hasattr(
            self.estimator,
            "decision_function",
        ):
            return np.asarray(
                self.estimator.decision_function(
                    Xs
                )
            ).reshape(
                -1
            )

        return self.estimator.predict_proba(
            Xs
        )[
            :,
            1,
        ]

    def size_bytes(
        self,
    ):
        return len(
            pickle.dumps({
                "scaler": self.scaler,
                "estimator": self.estimator,
            })
        )

    def save(
        self,
        path,
    ):
        joblib.dump(
            {
                "family": self.family,
                "scaler": self.scaler,
                "estimator": self.estimator,
            },
            path,
        )

## 6. Neural architectures

In [6]:
class FlatMLP(
    nn.Module
):
    def __init__(
        self,
        input_dim=695,
    ):
        super().__init__()

        cfg = MODEL_CONFIG[
            "MLP"
        ]

        self.net = nn.Sequential(
            nn.Linear(
                input_dim,
                cfg[
                    "hidden_1"
                ],
            ),
            nn.ReLU(),
            nn.Dropout(
                cfg[
                    "dropout"
                ]
            ),
            nn.Linear(
                cfg[
                    "hidden_1"
                ],
                cfg[
                    "hidden_2"
                ],
            ),
            nn.ReLU(),
            nn.Dropout(
                cfg[
                    "dropout"
                ]
            ),
            nn.Linear(
                cfg[
                    "hidden_2"
                ],
                1,
            ),
        )

    def forward(
        self,
        x,
    ):
        return self.net(
            x
        ).squeeze(
            1
        )


class AcousticSequenceModel(
    nn.Module
):
    def __init__(
        self,
        family,
    ):
        super().__init__()

        self.family = family

        common = MODEL_CONFIG[
            "deep_common"
        ]

        channels = common[
            "conv_channels"
        ]

        self.conv = nn.Sequential(
            nn.Conv1d(
                96,
                channels,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.ReLU(),
            nn.Conv1d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.ReLU(),
        )

        temporal_dim = channels

        if family == "CNN":
            self.temporal = None

        elif family == "CNN + LSTM":
            hidden = MODEL_CONFIG[
                family
            ][
                "hidden"
            ]

            self.temporal = nn.LSTM(
                input_size=channels,
                hidden_size=hidden,
                batch_first=True,
                bidirectional=False,
            )

            temporal_dim = hidden

        elif family == "CNN + BiLSTM":
            hidden = MODEL_CONFIG[
                family
            ][
                "hidden_per_direction"
            ]

            self.temporal = nn.LSTM(
                input_size=channels,
                hidden_size=hidden,
                batch_first=True,
                bidirectional=True,
            )

            temporal_dim = (
                hidden
                * 2
            )

        elif family == "CNN + BiGRU":
            hidden = MODEL_CONFIG[
                family
            ][
                "hidden_per_direction"
            ]

            self.temporal = nn.GRU(
                input_size=channels,
                hidden_size=hidden,
                batch_first=True,
                bidirectional=True,
            )

            temporal_dim = (
                hidden
                * 2
            )

        elif family == "CNN + Transformer":
            cfg = MODEL_CONFIG[
                family
            ]

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=channels,
                nhead=cfg[
                    "heads"
                ],
                dim_feedforward=cfg[
                    "feedforward"
                ],
                dropout=common[
                    "dropout"
                ],
                batch_first=True,
                norm_first=True,
            )

            self.temporal = nn.TransformerEncoder(
                encoder_layer,
                num_layers=cfg[
                    "layers"
                ],
            )

            temporal_dim = channels

        else:
            raise ValueError(
                family
            )

        self.global_branch = nn.Sequential(
            nn.Linear(
                119,
                common[
                    "global_hidden"
                ],
            ),
            nn.ReLU(),
            nn.Dropout(
                common[
                    "dropout"
                ]
            ),
        )

        combined_dim = (
            temporal_dim
            + common[
                "global_hidden"
            ]
        )

        self.head = nn.Sequential(
            nn.Linear(
                combined_dim,
                common[
                    "head_hidden"
                ],
            ),
            nn.ReLU(),
            nn.Dropout(
                common[
                    "dropout"
                ]
            ),
            nn.Linear(
                common[
                    "head_hidden"
                ],
                1,
            ),
        )

    def forward(
        self,
        seq,
        global_features,
    ):
        # seq: N x 6 x 96
        conv_out = self.conv(
            seq.transpose(
                1,
                2,
            )
        )

        temporal_input = conv_out.transpose(
            1,
            2,
        )

        if self.family == "CNN":
            temporal_vector = temporal_input.mean(
                dim=1
            )

        elif self.family in [
            "CNN + LSTM",
            "CNN + BiLSTM",
            "CNN + BiGRU",
        ]:
            temporal_output, _ = self.temporal(
                temporal_input
            )

            temporal_vector = temporal_output[
                :,
                -1,
                :,
            ]

        elif self.family == "CNN + Transformer":
            temporal_output = self.temporal(
                temporal_input
            )

            temporal_vector = temporal_output.mean(
                dim=1
            )

        else:
            raise ValueError(
                self.family
            )

        global_vector = self.global_branch(
            global_features
        )

        combined = torch.cat(
            [
                temporal_vector,
                global_vector,
            ],
            dim=1,
        )

        return self.head(
            combined
        ).squeeze(
            1
        )

## 7. Neural training helpers with password-grouped early stopping

In [7]:
def make_pos_weight(
    y,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    n_pos = float(
        np.sum(
            y
            == 1
        )
    )

    n_neg = float(
        np.sum(
            y
            == 0
        )
    )

    return (
        n_neg
        / max(
            n_pos,
            1.0,
        )
    )


def candidate_bce_loss(
    logits,
    y,
    pos_weight,
):
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [
                pos_weight
            ],
            dtype=torch.float32,
            device=logits.device,
        )
    )

    return criterion(
        logits,
        y,
    )


def val_fused_top1(
    meta_frame,
    scores,
):
    temp = meta_frame.copy()
    temp[
        "score"
    ] = np.asarray(
        scores,
        dtype=float,
    )

    # within-decision score normalisation
    temp[
        "score_z"
    ] = np.nan

    for (
        decision_uid,
        inds,
    ) in temp.groupby(
        "decision_uid"
    ).groups.items():
        inds = np.array(
            list(
                inds
            ),
            dtype=int,
        )

        vals = temp.loc[
            inds,
            "score",
        ].to_numpy(
            dtype=float
        )

        sd = float(
            np.std(
                vals
            )
        )

        scale = (
            sd
            if sd > 1e-12
            else 1.0
        )

        temp.loc[
            inds,
            "score_z",
        ] = (
            vals
            - np.mean(
                vals
            )
        ) / scale

    fused = (
        temp.groupby(
            [
                "profile_id",
                "wheel",
                "direction",
                "to_digit",
                "true_digit",
            ],
            as_index=False,
        )
        .agg(
            score=(
                "score_z",
                "mean",
            ),
            repeats=(
                "repeat_id",
                "nunique",
            ),
        )
    )

    assert fused[
        "repeats"
    ].eq(
        2
    ).all()

    hits = []

    for (
        profile_id,
        wheel,
        direction,
    ), group in fused.groupby(
        [
            "profile_id",
            "wheel",
            "direction",
        ],
        sort=False,
    ):
        ranked = group.sort_values(
            [
                "score",
                "to_digit",
            ],
            ascending=[
                False,
                True,
            ],
            kind="mergesort",
        )

        truth = int(
            ranked[
                "true_digit"
            ].iloc[
                0
            ]
        )

        pred = int(
            ranked[
                "to_digit"
            ].iloc[
                0
            ]
        )

        hits.append(
            int(
                pred
                == truth
            )
        )

    return float(
        np.mean(
            hits
        )
    )


def deterministic_inner_val_batch(
    heldout_batch,
):
    """
    Pick exactly one of the six outer-training passwords for
    neural early stopping, using a deterministic cyclic rule.
    """

    outer_index = BATCHES.index(
        heldout_batch
    )

    candidate = BATCHES[
        (
            outer_index
            + 1
        )
        % len(
            BATCHES
        )
    ]

    if candidate == heldout_batch:
        candidate = BATCHES[
            (
                outer_index
                + 2
            )
            % len(
                BATCHES
            )
        ]

    return candidate


def neural_family_config(
    family,
):
    if family == "MLP":
        return MODEL_CONFIG[
            "MLP"
        ]

    return MODEL_CONFIG[
        "deep_common"
    ]


def build_neural_network(
    family,
):
    if family == "MLP":
        return FlatMLP(
            input_dim=695
        )

    return AcousticSequenceModel(
        family=family
    )


def prepare_neural_scalers(
    family,
    train_idx,
):
    if family == "MLP":
        flat_scaler = StandardScaler()

        flat_scaler.fit(
            X[
                train_idx
            ]
        )

        return {
            "flat": flat_scaler,
        }

    seq_scaler = StandardScaler()
    global_scaler = StandardScaler()

    seq_train = X_seq[
        train_idx
    ].reshape(
        -1,
        96,
    )

    seq_scaler.fit(
        seq_train
    )

    global_scaler.fit(
        X_global[
            train_idx
        ]
    )

    return {
        "seq": seq_scaler,
        "global": global_scaler,
    }


def transform_neural(
    family,
    indices,
    scalers,
):
    if family == "MLP":
        flat = scalers[
            "flat"
        ].transform(
            X[
                indices
            ]
        ).astype(
            np.float32
        )

        return (
            flat,
            None,
        )

    seq = X_seq[
        indices
    ].copy()

    seq_shape = seq.shape

    seq = scalers[
        "seq"
    ].transform(
        seq.reshape(
            -1,
            96,
        )
    ).reshape(
        seq_shape
    ).astype(
        np.float32
    )

    global_features = scalers[
        "global"
    ].transform(
        X_global[
            indices
        ]
    ).astype(
        np.float32
    )

    return (
        seq,
        global_features,
    )


def neural_forward(
    family,
    model,
    batch_a,
    batch_b=None,
):
    if family == "MLP":
        return model(
            batch_a
        )

    return model(
        batch_a,
        batch_b,
    )


def make_neural_loader(
    family,
    indices,
    scalers,
    batch_size,
    shuffle,
    seed,
):
    a, b = transform_neural(
        family,
        indices,
        scalers,
    )

    y = manifest.loc[
        indices,
        "y",
    ].to_numpy(
        dtype=np.float32
    )

    if family == "MLP":
        dataset = TensorDataset(
            torch.from_numpy(
                a
            ),
            torch.from_numpy(
                y
            ),
        )
    else:
        dataset = TensorDataset(
            torch.from_numpy(
                a
            ),
            torch.from_numpy(
                b
            ),
            torch.from_numpy(
                y
            ),
        )

    generator = torch.Generator()
    generator.manual_seed(
        int(
            seed
        )
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        drop_last=False,
    )


def predict_neural_scores(
    family,
    model,
    indices,
    scalers,
):
    cfg = neural_family_config(
        family
    )

    loader = make_neural_loader(
        family=family,
        indices=indices,
        scalers=scalers,
        batch_size=cfg[
            "batch_size"
        ],
        shuffle=False,
        seed=SEED,
    )

    model.eval()

    scores = []

    with torch.no_grad():
        for batch in loader:
            if family == "MLP":
                xb, _ = batch

                logits = neural_forward(
                    family,
                    model,
                    xb.to(
                        DEVICE
                    ),
                )
            else:
                seq_b, global_b, _ = batch

                logits = neural_forward(
                    family,
                    model,
                    seq_b.to(
                        DEVICE
                    ),
                    global_b.to(
                        DEVICE
                    ),
                )

            scores.append(
                logits.detach().cpu().numpy()
            )

    return np.concatenate(
        scores
    ).reshape(
        -1
    )


def train_neural_fixed_epochs(
    family,
    train_idx,
    epochs,
    seed,
):
    random.seed(
        int(
            seed
        )
    )
    np.random.seed(
        int(
            seed
        )
    )
    torch.manual_seed(
        int(
            seed
        )
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )

    cfg = neural_family_config(
        family
    )

    scalers = prepare_neural_scalers(
        family,
        train_idx,
    )

    loader = make_neural_loader(
        family=family,
        indices=train_idx,
        scalers=scalers,
        batch_size=cfg[
            "batch_size"
        ],
        shuffle=True,
        seed=seed,
    )

    model = build_neural_network(
        family
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg[
            "learning_rate"
        ],
        weight_decay=cfg[
            "weight_decay"
        ],
    )

    pos_weight = make_pos_weight(
        manifest.loc[
            train_idx,
            "y",
        ].to_numpy(
            dtype=int
        )
    )

    train_loss_history = []

    for epoch in range(
        int(
            epochs
        )
    ):
        model.train()

        running_loss = 0.0
        n_seen = 0

        for batch in loader:
            optimizer.zero_grad(
                set_to_none=True
            )

            if family == "MLP":
                xb, yb = batch

                xb = xb.to(
                    DEVICE
                )

                yb = yb.to(
                    DEVICE
                )

                logits = neural_forward(
                    family,
                    model,
                    xb,
                )

            else:
                seq_b, global_b, yb = batch

                seq_b = seq_b.to(
                    DEVICE
                )

                global_b = global_b.to(
                    DEVICE
                )

                yb = yb.to(
                    DEVICE
                )

                logits = neural_forward(
                    family,
                    model,
                    seq_b,
                    global_b,
                )

            loss = candidate_bce_loss(
                logits,
                yb,
                pos_weight,
            )

            loss.backward()
            optimizer.step()

            batch_n = int(
                yb.shape[
                    0
                ]
            )

            running_loss += (
                float(
                    loss.detach().cpu()
                )
                * batch_n
            )

            n_seen += batch_n

        train_loss_history.append(
            running_loss
            / max(
                n_seen,
                1,
            )
        )

    return (
        model,
        scalers,
        train_loss_history,
    )


def select_neural_epoch(
    family,
    inner_train_idx,
    val_idx,
    seed,
):
    cfg = neural_family_config(
        family
    )

    random.seed(
        int(
            seed
        )
    )
    np.random.seed(
        int(
            seed
        )
    )
    torch.manual_seed(
        int(
            seed
        )
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )

    scalers = prepare_neural_scalers(
        family,
        inner_train_idx,
    )

    train_loader = make_neural_loader(
        family=family,
        indices=inner_train_idx,
        scalers=scalers,
        batch_size=cfg[
            "batch_size"
        ],
        shuffle=True,
        seed=seed,
    )

    val_loader = make_neural_loader(
        family=family,
        indices=val_idx,
        scalers=scalers,
        batch_size=cfg[
            "batch_size"
        ],
        shuffle=False,
        seed=seed,
    )

    model = build_neural_network(
        family
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg[
            "learning_rate"
        ],
        weight_decay=cfg[
            "weight_decay"
        ],
    )

    pos_weight = make_pos_weight(
        manifest.loc[
            inner_train_idx,
            "y",
        ].to_numpy(
            dtype=int
        )
    )

    best_val_loss = np.inf
    best_epoch = 1
    epochs_without_improvement = 0
    history = []

    for epoch in range(
        1,
        cfg[
            "max_epochs"
        ]
        + 1,
    ):
        model.train()

        train_running = 0.0
        train_seen = 0

        for batch in train_loader:
            optimizer.zero_grad(
                set_to_none=True
            )

            if family == "MLP":
                xb, yb = batch

                xb = xb.to(
                    DEVICE
                )

                yb = yb.to(
                    DEVICE
                )

                logits = neural_forward(
                    family,
                    model,
                    xb,
                )
            else:
                seq_b, global_b, yb = batch

                seq_b = seq_b.to(
                    DEVICE
                )

                global_b = global_b.to(
                    DEVICE
                )

                yb = yb.to(
                    DEVICE
                )

                logits = neural_forward(
                    family,
                    model,
                    seq_b,
                    global_b,
                )

            loss = candidate_bce_loss(
                logits,
                yb,
                pos_weight,
            )

            loss.backward()
            optimizer.step()

            batch_n = int(
                yb.shape[
                    0
                ]
            )

            train_running += (
                float(
                    loss.detach().cpu()
                )
                * batch_n
            )

            train_seen += batch_n

        train_loss = (
            train_running
            / max(
                train_seen,
                1,
            )
        )

        model.eval()

        val_running = 0.0
        val_seen = 0
        val_scores = []

        with torch.no_grad():
            for batch in val_loader:
                if family == "MLP":
                    xb, yb = batch

                    xb = xb.to(
                        DEVICE
                    )

                    yb = yb.to(
                        DEVICE
                    )

                    logits = neural_forward(
                        family,
                        model,
                        xb,
                    )

                else:
                    seq_b, global_b, yb = batch

                    seq_b = seq_b.to(
                        DEVICE
                    )

                    global_b = global_b.to(
                        DEVICE
                    )

                    yb = yb.to(
                        DEVICE
                    )

                    logits = neural_forward(
                        family,
                        model,
                        seq_b,
                        global_b,
                    )

                loss = candidate_bce_loss(
                    logits,
                    yb,
                    pos_weight,
                )

                batch_n = int(
                    yb.shape[
                        0
                    ]
                )

                val_running += (
                    float(
                        loss.detach().cpu()
                    )
                    * batch_n
                )

                val_seen += batch_n

                val_scores.append(
                    logits.detach().cpu().numpy()
                )

        val_loss = (
            val_running
            / max(
                val_seen,
                1,
            )
        )

        val_scores = np.concatenate(
            val_scores
        ).reshape(
            -1
        )

        val_top1 = val_fused_top1(
            manifest.loc[
                val_idx
            ].reset_index(
                drop=True
            ),
            val_scores,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_fused_top1": val_top1,
        })

        if (
            val_loss
            < best_val_loss
            - 1e-5
        ):
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= cfg[
                "patience"
            ]
        ):
            break

    history = pd.DataFrame(
        history
    )

    return (
        int(
            best_epoch
        ),
        history,
    )

## 8. Common ranking and A/B fusion

In [8]:
def within_decision_z(
    frame,
    score_col="score",
):
    frame = (
        frame
        .reset_index(
            drop=True
        )
        .copy()
    )

    frame[
        "score_z"
    ] = np.nan

    for (
        decision_uid,
        inds,
    ) in frame.groupby(
        "decision_uid"
    ).groups.items():
        inds = np.array(
            list(
                inds
            ),
            dtype=int,
        )

        values = frame.loc[
            inds,
            score_col,
        ].to_numpy(
            dtype=float
        )

        sd = float(
            np.std(
                values
            )
        )

        scale = (
            sd
            if sd > 1e-12
            else 1.0
        )

        frame.loc[
            inds,
            "score_z",
        ] = (
            values
            - np.mean(
                values
            )
        ) / scale

    return frame


def fuse_ab_and_rank(
    candidate_frame,
    family,
):
    candidate_frame = within_decision_z(
        candidate_frame,
        "score",
    )

    fused_candidates = (
        candidate_frame.groupby(
            [
                "batch",
                "password",
                "profile_id",
                "wheel",
                "direction",
                "to_digit",
                "true_digit",
            ],
            as_index=False,
        )
        .agg(
            fused_score=(
                "score_z",
                "mean",
            ),
            n_repeats=(
                "repeat_id",
                "nunique",
            ),
            y=(
                "y",
                "first",
            ),
        )
    )

    assert fused_candidates[
        "n_repeats"
    ].eq(
        2
    ).all()

    rows = []

    for (
        batch,
        password,
        profile_id,
        wheel,
        direction,
    ), group in fused_candidates.groupby(
        [
            "batch",
            "password",
            "profile_id",
            "wheel",
            "direction",
        ],
        sort=False,
    ):
        assert len(
            group
        ) == 10

        ranked = (
            group
            .sort_values(
                [
                    "fused_score",
                    "to_digit",
                ],
                ascending=[
                    False,
                    True,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        truth = int(
            ranked[
                "true_digit"
            ].iloc[
                0
            ]
        )

        digits = (
            ranked[
                "to_digit"
            ]
            .astype(
                int
            )
            .tolist()
        )

        true_rank = int(
            digits.index(
                truth
            )
            + 1
        )

        rows.append({
            "family": family,
            "heldout_batch": batch,
            "password": str(
                password
            ),
            "profile_id": profile_id,
            "wheel": int(
                wheel
            ),
            "direction": direction,
            "true_digit": truth,
            "pred_digit": int(
                digits[
                    0
                ]
            ),
            "true_rank": true_rank,
            "top1": int(
                true_rank <= 1
            ),
            "top2": int(
                true_rank <= 2
            ),
            "top3": int(
                true_rank <= 3
            ),
            "reciprocal_rank": (
                1.0
                / true_rank
            ),
        })

    return (
        fused_candidates,
        pd.DataFrame(
            rows
        ),
    )


def metric_row(
    frame,
):
    return {
        "n_decisions": len(
            frame
        ),
        "top1": float(
            frame[
                "top1"
            ].mean()
        ),
        "top2": float(
            frame[
                "top2"
            ].mean()
        ),
        "top3": float(
            frame[
                "top3"
            ].mean()
        ),
        "mean_true_rank": float(
            frame[
                "true_rank"
            ].mean()
        ),
        "mrr": float(
            frame[
                "reciprocal_rank"
            ].mean()
        ),
        "macro_f1": float(
            f1_score(
                frame[
                    "true_digit"
                ],
                frame[
                    "pred_digit"
                ],
                labels=list(
                    range(
                        10
                    )
                ),
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                frame[
                    "true_digit"
                ],
                frame[
                    "pred_digit"
                ],
                labels=list(
                    range(
                        10
                    )
                ),
                average="weighted",
                zero_division=0,
            )
        ),
    }

## 9. Seven-fold password-LOPO model comparison

In [9]:
candidate_parts = []
fit_records = []
neural_history_records = []

NEURAL_FAMILIES = [
    "MLP",
] + DEEP_FAMILIES

for fold_index, heldout in enumerate(
    BATCHES
):
    print(
        f"\nOuter held-out password: {heldout}"
    )

    inner_val_batch = deterministic_inner_val_batch(
        heldout
    )

    print(
        "Inner validation password:",
        inner_val_batch,
    )

    for family_index, family in enumerate(
        MODEL_FAMILIES
    ):
        print(
            "  ",
            family,
        )

        for wheel in WHEELS:
            for direction in DIRECTIONS:
                domain_mask = (
                    manifest[
                        "wheel"
                    ].eq(
                        wheel
                    )
                    &
                    manifest[
                        "direction"
                    ].eq(
                        direction
                    )
                )

                outer_train_mask = (
                    domain_mask
                    &
                    manifest[
                        "batch"
                    ].ne(
                        heldout
                    )
                )

                outer_test_mask = (
                    domain_mask
                    &
                    manifest[
                        "batch"
                    ].eq(
                        heldout
                    )
                )

                outer_train_idx = manifest.index[
                    outer_train_mask
                ].to_numpy()

                outer_test_idx = manifest.index[
                    outer_test_mask
                ].to_numpy()

                assert len(
                    outer_train_idx
                ) == 1200

                assert len(
                    outer_test_idx
                ) == 200

                seed = (
                    SEED
                    + fold_index
                    * 10000
                    + family_index
                    * 1000
                    + wheel
                    * 10
                    + (
                        1
                        if direction
                        == "CW"
                        else 0
                    )
                )

                started = time.perf_counter()

                if family in [
                    "Logistic Regression",
                    "RBF-SVM",
                    "HistGradientBoosting",
                ]:
                    model = ClassicalScoreModel(
                        family=family,
                        seed=seed,
                    )

                    model.fit(
                        X[
                            outer_train_idx
                        ],
                        manifest.loc[
                            outer_train_idx,
                            "y",
                        ].to_numpy(
                            dtype=int
                        ),
                    )

                    score = model.score(
                        X[
                            outer_test_idx
                        ]
                    )

                    selected_epoch = np.nan
                    model_size = model.size_bytes()

                else:
                    inner_train_mask = (
                        outer_train_mask
                        &
                        manifest[
                            "batch"
                        ].ne(
                            inner_val_batch
                        )
                    )

                    inner_val_mask = (
                        outer_train_mask
                        &
                        manifest[
                            "batch"
                        ].eq(
                            inner_val_batch
                        )
                    )

                    inner_train_idx = manifest.index[
                        inner_train_mask
                    ].to_numpy()

                    inner_val_idx = manifest.index[
                        inner_val_mask
                    ].to_numpy()

                    assert len(
                        inner_train_idx
                    ) == 1000

                    assert len(
                        inner_val_idx
                    ) == 200

                    selected_epoch, history = select_neural_epoch(
                        family=family,
                        inner_train_idx=inner_train_idx,
                        val_idx=inner_val_idx,
                        seed=seed,
                    )

                    history[
                        "heldout_batch"
                    ] = heldout

                    history[
                        "inner_val_batch"
                    ] = inner_val_batch

                    history[
                        "family"
                    ] = family

                    history[
                        "wheel"
                    ] = wheel

                    history[
                        "direction"
                    ] = direction

                    history[
                        "selected_epoch"
                    ] = selected_epoch

                    neural_history_records.append(
                        history
                    )

                    # Retrain from scratch on all six outer-training passwords
                    # for exactly the inner-selected epoch count.
                    final_seed = (
                        seed
                        + 500000
                    )

                    model, scalers, _ = train_neural_fixed_epochs(
                        family=family,
                        train_idx=outer_train_idx,
                        epochs=selected_epoch,
                        seed=final_seed,
                    )

                    score = predict_neural_scores(
                        family=family,
                        model=model,
                        indices=outer_test_idx,
                        scalers=scalers,
                    )

                    buffer = io.BytesIO()

                    torch.save(
                        model.state_dict(),
                        buffer,
                    )

                    model_size = len(
                        buffer.getvalue()
                    )

                    if family == "MLP":
                        model_size += (
                            scalers[
                                "flat"
                            ].mean_.nbytes
                            + scalers[
                                "flat"
                            ].scale_.nbytes
                        )
                    else:
                        model_size += (
                            scalers[
                                "seq"
                            ].mean_.nbytes
                            + scalers[
                                "seq"
                            ].scale_.nbytes
                            + scalers[
                                "global"
                            ].mean_.nbytes
                            + scalers[
                                "global"
                            ].scale_.nbytes
                        )

                fit_seconds = (
                    time.perf_counter()
                    - started
                )

                part = manifest.loc[
                    outer_test_idx
                ].copy()

                part[
                    "family"
                ] = family

                part[
                    "score"
                ] = np.asarray(
                    score,
                    dtype=float,
                )

                candidate_parts.append(
                    part
                )

                fit_records.append({
                    "heldout_batch": heldout,
                    "inner_val_batch": (
                        inner_val_batch
                        if family in NEURAL_FAMILIES
                        else ""
                    ),
                    "family": family,
                    "wheel": wheel,
                    "direction": direction,
                    "n_outer_train_candidates": len(
                        outer_train_idx
                    ),
                    "selected_epoch": (
                        selected_epoch
                    ),
                    "fit_seconds": fit_seconds,
                    "model_size_bytes": model_size,
                })

                del model

                if family in NEURAL_FAMILIES:
                    try:
                        del scalers
                    except Exception:
                        pass

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()


candidate_scores = pd.concat(
    candidate_parts,
    ignore_index=True,
)

fit_records = pd.DataFrame(
    fit_records
)

if neural_history_records:
    neural_history = pd.concat(
        neural_history_records,
        ignore_index=True,
    )
else:
    neural_history = pd.DataFrame()


expected_rows = (
    len(
        MODEL_FAMILIES
    )
    * len(
        manifest
    )
)

assert len(
    candidate_scores
) == expected_rows


fused_candidate_parts = []
decision_parts = []

for family in MODEL_FAMILIES:
    fc, family_decisions = fuse_ab_and_rank(
        candidate_scores[
            candidate_scores[
                "family"
            ]
            == family
        ].copy(),
        family=family,
    )

    fc[
        "family"
    ] = family

    fused_candidate_parts.append(
        fc
    )

    decision_parts.append(
        family_decisions
    )


fused_candidates = pd.concat(
    fused_candidate_parts,
    ignore_index=True,
)

decisions = pd.concat(
    decision_parts,
    ignore_index=True,
)

assert len(
    decisions
) == (
    len(
        MODEL_FAMILIES
    )
    * 420
)

print(
    "\nLOPO model-family comparison complete."
)


Outer held-out password: B01_2085
Inner validation password: B02_4369
   Logistic Regression
   RBF-SVM
   HistGradientBoosting
   MLP
   CNN
   CNN + LSTM
   CNN + BiLSTM
   CNN + BiGRU
   CNN + Transformer


/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm


Outer held-out password: B02_4369
Inner validation password: B03_5720
   Logistic Regression
   RBF-SVM
   HistGradientBoosting
   MLP
   CNN
   CNN + LSTM
   CNN + BiLSTM
   CNN + BiGRU
   CNN + Transformer


/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEnco


Outer held-out password: B03_5720
Inner validation password: B04_6893
   Logistic Regression
   RBF-SVM
   HistGradientBoosting
   MLP
   CNN
   CNN + LSTM
   CNN + BiLSTM
   CNN + BiGRU
   CNN + Transformer


...


Outer held-out password: B06_8451
Inner validation password: B07_9638
   Logistic Regression
   RBF-SVM
   HistGradientBoosting
   MLP
   CNN
   CNN + LSTM
   CNN + BiLSTM
   CNN + BiGRU
   CNN + Transformer


/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEnco


Outer held-out password: B07_9638
Inner validation password: B01_2085
   Logistic Regression
   RBF-SVM
   HistGradientBoosting
   MLP
   CNN
   CNN + LSTM
   CNN + BiLSTM
   CNN + BiGRU
   CNN + Transformer


/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEncoder(
/tmp/ipykernel_2077/4015209727.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal = nn.TransformerEnco


LOPO model-family comparison complete.


## 10. Performance summaries

In [10]:
summary_rows = []
password_rows = []
wheel_rows = []
direction_rows = []

for family, group in decisions.groupby(
    "family",
    sort=False,
):
    summary_rows.append({
        "family": family,
        **metric_row(
            group
        ),
    })

    for wheel in WHEELS:
        wheel_rows.append({
            "family": family,
            "wheel": wheel,
            **metric_row(
                group[
                    group[
                        "wheel"
                    ]
                    == wheel
                ]
            ),
        })

    for direction in DIRECTIONS:
        direction_rows.append({
            "family": family,
            "direction": direction,
            **metric_row(
                group[
                    group[
                        "direction"
                    ]
                    == direction
                ]
            ),
        })

    for (
        heldout_batch,
        password,
    ), pg in group.groupby(
        [
            "heldout_batch",
            "password",
        ],
        sort=False,
    ):
        password_rows.append({
            "family": family,
            "heldout_batch": heldout_batch,
            "password": str(
                password
            ),
            **metric_row(
                pg
            ),
        })


summary = pd.DataFrame(
    summary_rows
)

wheel_summary = pd.DataFrame(
    wheel_rows
)

direction_summary = pd.DataFrame(
    direction_rows
)

password_summary = pd.DataFrame(
    password_rows
)


complexity_summary = (
    fit_records.groupby(
        "family",
        as_index=False,
    )
    .agg(
        mean_fit_seconds=(
            "fit_seconds",
            "mean",
        ),
        median_fit_seconds=(
            "fit_seconds",
            "median",
        ),
        mean_model_size_bytes=(
            "model_size_bytes",
            "mean",
        ),
        median_selected_epoch=(
            "selected_epoch",
            "median",
        ),
    )
)

summary = summary.merge(
    complexity_summary,
    on="family",
    how="left",
    validate="one_to_one",
)

display(
    summary.sort_values(
        "top1",
        ascending=False,
    ).round(
        4
    )
)

,family,n_decisions,top1,top2,top3,mean_true_rank,mrr,macro_f1,weighted_f1,mean_fit_seconds,median_fit_seconds,mean_model_size_bytes,median_selected_epoch
0,Logistic Regression,420,0.8595,0.9548,0.9786,1.2286,0.9195,0.8601,0.8591,0.2327,0.2276,2.327700e+04,NaN
6,CNN + BiLSTM,420,0.8071,0.9429,0.9667,1.3095,0.8902,0.8145,0.8080,1.5863,1.5438,8.877640e+05,6.0
4,CNN,420,0.8048,0.9357,0.9690,1.3143,0.8882,0.8090,0.8039,1.6544,1.7042,4.893670e+05,9.0
7,CNN + BiGRU,420,0.7976,0.9262,0.9619,1.3476,0.8820,0.7988,0.7970,1.6086,1.4863,7.884360e+05,6.0
1,RBF-SVM,420,0.7857,0.9238,0.9738,1.3452,0.8768,0.7912,0.7865,0.2865,0.2737,3.247318e+06,NaN
5,CNN + LSTM,420,0.7714,0.9190,0.9667,1.3738,0.8683,0.7738,0.7708,1.7003,1.4985,1.018324e+06,6.0
8,CNN + Transformer,420,0.7571,0.9000,0.9476,1.4571,0.8552,0.7617,0.7561,1.5006,1.4207,1.023667e+06,4.0
2,HistGradientBoosting,420,0.7214,0.8786,0.9405,1.5095,0.8335,0.7166,0.7225,6.7631,6.7990,1.824113e+06,NaN
3,MLP,420,0.6786,0.8333,0.9143,1.6714,0.8006,0.6820,0.6779,0.7681,0.7256,8.587570e+05,4.0


## 11. Classification reports and confusion matrices for every model

In [11]:
REPORT_DIR = (
    RESULT_DIR
    / "model_reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

all_digits = list(
    range(
        10
    )
)

report_summary_rows = []
report_per_class_rows = []


def family_slug(
    family,
):
    return (
        family.lower()
        .replace(
            " + ",
            "_plus_",
        )
        .replace(
            " ",
            "_",
        )
        .replace(
            "-",
            "_",
        )
    )


for family in MODEL_FAMILIES:
    frame = decisions[
        decisions[
            "family"
        ]
        == family
    ]

    report = classification_report(
        frame[
            "true_digit"
        ],
        frame[
            "pred_digit"
        ],
        labels=all_digits,
        target_names=[
            str(
                i
            )
            for i in all_digits
        ],
        output_dict=True,
        zero_division=0,
    )

    slug = family_slug(
        family
    )

    with open(
        REPORT_DIR
        / f"{slug}_classification_report.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            report,
            f,
            indent=2,
        )

    for digit in all_digits:
        row = report[
            str(
                digit
            )
        ]

        report_per_class_rows.append({
            "family": family,
            "digit": digit,
            "precision": float(
                row[
                    "precision"
                ]
            ),
            "recall": float(
                row[
                    "recall"
                ]
            ),
            "f1_score": float(
                row[
                    "f1-score"
                ]
            ),
            "support": int(
                row[
                    "support"
                ]
            ),
        })

    report_summary_rows.append({
        "family": family,
        "accuracy": float(
            report[
                "accuracy"
            ]
        ),
        "macro_precision": float(
            report[
                "macro avg"
            ][
                "precision"
            ]
        ),
        "macro_recall": float(
            report[
                "macro avg"
            ][
                "recall"
            ]
        ),
        "macro_f1": float(
            report[
                "macro avg"
            ][
                "f1-score"
            ]
        ),
        "weighted_f1": float(
            report[
                "weighted avg"
            ][
                "f1-score"
            ]
        ),
    })

    cm = confusion_matrix(
        frame[
            "true_digit"
        ],
        frame[
            "pred_digit"
        ],
        labels=all_digits,
    )

    pd.DataFrame(
        cm,
        index=all_digits,
        columns=all_digits,
    ).to_csv(
        REPORT_DIR
        / f"{slug}_confusion_matrix.csv"
    )

    fig, ax = plt.subplots(
        figsize=(
            6.3,
            5.2,
        )
    )

    image = ax.imshow(
        cm,
        cmap="Blues",
        aspect="equal",
        vmin=0,
        vmax=max(
            int(
                cm.max()
            ),
            1,
        ),
    )

    ax.set_xticks(
        np.arange(
            10
        )
    )

    ax.set_xticklabels(
        all_digits
    )

    ax.set_yticks(
        np.arange(
            10
        )
    )

    ax.set_yticklabels(
        all_digits
    )

    ax.set_xlabel(
        "Predicted digit"
    )

    ax.set_ylabel(
        "True digit"
    )

    threshold = (
        cm.max()
        / 2
        if cm.max()
        > 0
        else 0
    )

    for i in range(
        10
    ):
        for j in range(
            10
        ):
            value = int(
                cm[
                    i,
                    j,
                ]
            )

            ax.text(
                j,
                i,
                str(
                    value
                ),
                ha="center",
                va="center",
                fontsize=7.5,
                color=(
                    "white"
                    if value
                    > threshold
                    else "black"
                ),
            )

    cbar = fig.colorbar(
        image,
        ax=ax,
        shrink=0.85,
    )

    cbar.set_label(
        "LOPO decisions"
    )

    fig.tight_layout()

    fig.savefig(
        REPORT_DIR
        / f"{slug}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        REPORT_DIR
        / f"{slug}_confusion_matrix.pdf",
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


classification_report_summary = pd.DataFrame(
    report_summary_rows
)

classification_report_per_digit = pd.DataFrame(
    report_per_class_rows
)

display(
    classification_report_summary.round(
        4
    )
)

,family,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,Logistic Regression,0.8595,0.8582,0.8642,0.8601,0.8591
1,RBF-SVM,0.7857,0.7965,0.7917,0.7912,0.7865
2,HistGradientBoosting,0.7214,0.7228,0.7133,0.7166,0.7225
3,MLP,0.6786,0.6832,0.6892,0.6820,0.6779
4,CNN,0.8048,0.8102,0.8117,0.8090,0.8039
5,CNN + LSTM,0.7714,0.7764,0.7767,0.7738,0.7708
6,CNN + BiLSTM,0.8071,0.8170,0.8158,0.8145,0.8080
7,CNN + BiGRU,0.7976,0.8010,0.8017,0.7988,0.7970
8,CNN + Transformer,0.7571,0.7619,0.7708,0.7617,0.7561


## 12. Password-level paired comparisons

In [12]:
password_top1 = (
    password_summary.pivot(
        index=[
            "heldout_batch",
            "password",
        ],
        columns="family",
        values="top1",
    )
    .reset_index()
)

baseline_family = (
    "Logistic Regression"
)

pairwise_rows = []

for family in MODEL_FAMILIES:
    if family == baseline_family:
        continue

    delta = (
        password_top1[
            family
        ]
        - password_top1[
            baseline_family
        ]
    )

    test = wilcoxon(
        password_top1[
            family
        ],
        password_top1[
            baseline_family
        ],
        alternative="two-sided",
        method="auto",
    )

    pairwise_rows.append({
        "comparison": (
            f"{family} vs {baseline_family}"
        ),
        "mean_delta_pp": float(
            100
            * delta.mean()
        ),
        "median_delta_pp": float(
            100
            * delta.median()
        ),
        "family_better_passwords": int(
            (
                delta
                > 0
            ).sum()
        ),
        "equal_passwords": int(
            (
                delta
                == 0
            ).sum()
        ),
        "baseline_better_passwords": int(
            (
                delta
                < 0
            ).sum()
        ),
        "p_two_sided": float(
            test.pvalue
        ),
    })


pairwise_tests = pd.DataFrame(
    pairwise_rows
)

display(
    pairwise_tests.round(
        4
    )
)

,comparison,mean_delta_pp,median_delta_pp,family_better_passwords,equal_passwords,baseline_better_passwords,p_two_sided
0,RBF-SVM vs Logistic Regression,-7.3810,-8.3333,1,0,6,0.0469
1,HistGradientBoosting vs Logistic Regression,-13.8095,-13.3333,0,0,7,0.0156
2,MLP vs Logistic Regression,-18.0952,-13.3333,0,0,7,0.0156
3,CNN vs Logistic Regression,-5.4762,-5.0000,1,2,4,0.1250
4,CNN + LSTM vs Logistic Regression,-8.8095,-8.3333,0,1,6,0.0312
5,CNN + BiLSTM vs Logistic Regression,-5.2381,-5.0000,0,0,7,0.0156
6,CNN + BiGRU vs Logistic Regression,-6.1905,-8.3333,1,0,6,0.0312
7,CNN + Transformer vs Logistic Regression,-10.2381,-10.0000,0,0,7,0.0156


## 13. Neural training dynamics

For MLP/CNN/RNN/Transformer families, inner-validation curves are retained.

The plots below show the mean train loss, mean validation loss and mean validation A/B-fused Top-1 across all outer-fold/domain training runs.

These curves are used to discuss:

- convergence speed;
- whether validation loss diverges from training loss;
- whether a higher-capacity architecture overfits;
- whether early stopping consistently occurs near the same epoch.

They are diagnostic only. Outer-password Top-1 remains the model-selection criterion.


In [13]:
NEURAL_HISTORY_DIR = (
    RESULT_DIR
    / "neural_training_curves"
)

NEURAL_HISTORY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not neural_history.empty:
    for family in NEURAL_FAMILIES:
        history = neural_history[
            neural_history[
                "family"
            ]
            == family
        ]

        if history.empty:
            continue

        curve = (
            history.groupby(
                "epoch"
            )
            .agg(
                train_loss=(
                    "train_loss",
                    "mean",
                ),
                val_loss=(
                    "val_loss",
                    "mean",
                ),
                val_top1=(
                    "val_fused_top1",
                    "mean",
                ),
            )
            .reset_index()
        )

        slug = family_slug(
            family
        )

        curve.to_csv(
            NEURAL_HISTORY_DIR
            / f"{slug}_mean_training_curve.csv",
            index=False,
        )

        fig, ax = plt.subplots(
            figsize=(
                7.0,
                4.0,
            )
        )

        ax.plot(
            curve[
                "epoch"
            ],
            curve[
                "train_loss"
            ],
            label="Train loss",
            linewidth=1.7,
        )

        ax.plot(
            curve[
                "epoch"
            ],
            curve[
                "val_loss"
            ],
            label="Validation loss",
            linewidth=1.7,
        )

        ax.set_xlabel(
            "Epoch"
        )

        ax.set_ylabel(
            "Weighted BCE loss"
        )

        ax.grid(
            axis="y",
            alpha=0.13,
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            NEURAL_HISTORY_DIR
            / f"{slug}_loss_curve.png",
            dpi=300,
            bbox_inches="tight",
        )

        fig.savefig(
            NEURAL_HISTORY_DIR
            / f"{slug}_loss_curve.pdf",
            bbox_inches="tight",
        )

        plt.close(
            fig
        )

        fig, ax = plt.subplots(
            figsize=(
                7.0,
                4.0,
            )
        )

        ax.plot(
            curve[
                "epoch"
            ],
            100
            * curve[
                "val_top1"
            ],
            linewidth=1.8,
        )

        ax.set_xlabel(
            "Epoch"
        )

        ax.set_ylabel(
            "Inner-validation A/B-fused Top-1"
        )

        ax.set_ylim(
            0,
            100,
        )

        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(
                lambda y, pos: (
                    f"{y:.0f}%"
                )
            )
        )

        ax.grid(
            axis="y",
            alpha=0.13,
        )

        fig.tight_layout()

        fig.savefig(
            NEURAL_HISTORY_DIR
            / f"{slug}_validation_top1.png",
            dpi=300,
            bbox_inches="tight",
        )

        fig.savefig(
            NEURAL_HISTORY_DIR
            / f"{slug}_validation_top1.pdf",
            bbox_inches="tight",
        )

        plt.close(
            fig
        )

print(
    "Neural training curves saved."
)

Neural training curves saved.


## 14. Deterministic model selection

In [14]:
selection_table = (
    summary[
        [
            "family",
            "top1",
            "top2",
            "top3",
            "mean_true_rank",
            "mrr",
            "macro_f1",
            "weighted_f1",
            "mean_fit_seconds",
            "mean_model_size_bytes",
            "median_selected_epoch",
        ]
    ]
    .sort_values(
        [
            "top1",
            "macro_f1",
            "mrr",
            "mean_model_size_bytes",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

SELECTED_FAMILY = str(
    selection_table.loc[
        0,
        "family",
    ]
)

print(
    "Selected family:",
    SELECTED_FAMILY,
)

display(
    selection_table.round(
        4
    )
)

Selected family: Logistic Regression


,family,top1,top2,top3,mean_true_rank,mrr,macro_f1,weighted_f1,mean_fit_seconds,mean_model_size_bytes,median_selected_epoch
0,Logistic Regression,0.8595,0.9548,0.9786,1.2286,0.9195,0.8601,0.8591,0.2327,2.327700e+04,NaN
1,CNN + BiLSTM,0.8071,0.9429,0.9667,1.3095,0.8902,0.8145,0.8080,1.5863,8.877640e+05,6.0
2,CNN,0.8048,0.9357,0.9690,1.3143,0.8882,0.8090,0.8039,1.6544,4.893670e+05,9.0
3,CNN + BiGRU,0.7976,0.9262,0.9619,1.3476,0.8820,0.7988,0.7970,1.6086,7.884360e+05,6.0
4,RBF-SVM,0.7857,0.9238,0.9738,1.3452,0.8768,0.7912,0.7865,0.2865,3.247318e+06,NaN
5,CNN + LSTM,0.7714,0.9190,0.9667,1.3738,0.8683,0.7738,0.7708,1.7003,1.018324e+06,6.0
6,CNN + Transformer,0.7571,0.9000,0.9476,1.4571,0.8552,0.7617,0.7561,1.5006,1.023667e+06,4.0
7,HistGradientBoosting,0.7214,0.8786,0.9405,1.5095,0.8335,0.7166,0.7225,6.7631,1.824113e+06,NaN
8,MLP,0.6786,0.8333,0.9143,1.6714,0.8006,0.6820,0.6779,0.7681,8.587570e+05,4.0


## 15. Report-ready figures

In [15]:
DARK_BLUE = "#315B7D"
MID_BLUE = "#6F8FA8"
LIGHT_BLUE = "#AFC5D5"
PALE_BLUE = "#DCE8F0"
MID_GREY = "#9EA5AA"
DARK_GREY = "#596168"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def save_figure(
    fig,
    stem,
):
    fig.savefig(
        RESULT_DIR
        / f"{stem}.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        RESULT_DIR
        / f"{stem}.pdf",
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

In [23]:
# Figure 1 — overall Top-1 and Macro-F1.

plot = (
    summary
    .set_index(
        "family"
    )
    .reindex(
        MODEL_FAMILIES
    )
)

x = np.arange(
    len(
        MODEL_FAMILIES
    )
)

width = 0.32

fig, ax = plt.subplots(
    figsize=(
        11.0,
        4.8,
    )
)

bars1 = ax.bar(
    x
    - width
    / 2,
    100
    * plot[
        "top1"
    ],
    width=width,
    color=DARK_BLUE,
    edgecolor="none",
    label="Top-1",
    zorder=3,
)

bars2 = ax.bar(
    x
    + width
    / 2,
    100
    * plot[
        "macro_f1"
    ],
    width=width,
    color=LIGHT_BLUE,
    edgecolor="none",
    label="Macro-F1",
    zorder=3,
)

for bars in [
    bars1,
    bars2,
]:
    for bar in bars:
        value = float(
            bar.get_height()
        )

        ax.annotate(
            f"{value:.1f}%",
            xy=(
                bar.get_x()
                + bar.get_width()
                / 2,
                value,
            ),
            xytext=(
                0,
                4,
            ),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            color=DARK_GREY,
        )


ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        "Logistic\nRegression",
        "RBF-SVM",
        "HistGradient\nBoosting",
        "MLP",
        "CNN",
        "CNN+\nLSTM",
        "CNN+\nBiLSTM",
        "CNN+\nBiGRU",
        "CNN+\nTransformer",
    ]
)

ax.set_ylabel(
    "Password-LOPO performance"
)

ax.set_ylim(
    0,
    100,
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda y, pos: (
            f"{y:.0f}%"
        )
    )
)

ax.grid(
    axis="y",
    alpha=0.13,
    zorder=0,
)

ax.legend(
    frameon=False,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        -0.15,
    ),
)

fig.subplots_adjust(
    bottom=0.27
)

fig.tight_layout()

save_figure(
    fig,
    "Final_Model_Fig1_Model_Comparison",
)

plt.show()

In [17]:
# Figure 2 — password × model Top-1 heatmap.

password_matrix = (
    password_summary.pivot(
        index="password",
        columns="family",
        values="top1",
    )
    .reindex(
        columns=MODEL_FAMILIES
    )
)

fig, ax = plt.subplots(
    figsize=(
        11.0,
        4.8,
    )
)

image = ax.imshow(
    100
    * password_matrix.to_numpy(),
    aspect="auto",
    cmap="Blues",
    vmin=40,
    vmax=100,
)

ax.set_xticks(
    np.arange(
        len(
            MODEL_FAMILIES
        )
    )
)

ax.set_xticklabels([
    "Logistic\nRegression",
    "RBF-SVM",
    "HistGradient\nBoosting",
    "MLP",
    "CNN",
    "CNN+\nLSTM",
    "CNN+\nBiLSTM",
    "CNN+\nBiGRU",
    "CNN+\nTransformer",
])

ax.set_yticks(
    np.arange(
        len(
            password_matrix
        )
    )
)

ax.set_yticklabels(
    password_matrix.index.astype(
        str
    )
)

ax.set_xlabel(
    "Model family"
)

ax.set_ylabel(
    "Held-out password"
)

for i in range(
    password_matrix.shape[
        0
    ]
):
    for j in range(
        password_matrix.shape[
            1
        ]
    ):
        value = (
            100
            * password_matrix.iloc[
                i,
                j,
            ]
        )

        ax.text(
            j,
            i,
            f"{value:.1f}%",
            ha="center",
            va="center",
            fontsize=7,
            color=(
                "white"
                if value
                >= 73
                else "black"
            ),
        )


cbar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.88,
)

cbar.set_label(
    "A/B-fused Top-1 accuracy"
)

fig.tight_layout()

save_figure(
    fig,
    "Final_Model_Fig2_Password_Heatmap",
)

plt.show()

In [18]:
# Figure 3 — selected-model confusion matrix.

selected_frame = decisions[
    decisions[
        "family"
    ]
    == SELECTED_FAMILY
]

selected_cm = confusion_matrix(
    selected_frame[
        "true_digit"
    ],
    selected_frame[
        "pred_digit"
    ],
    labels=list(
        range(
            10
        )
    ),
)

fig, ax = plt.subplots(
    figsize=(
        6.3,
        5.2,
    )
)

image = ax.imshow(
    selected_cm,
    cmap="Blues",
    aspect="equal",
    vmin=0,
    vmax=max(
        int(
            selected_cm.max()
        ),
        1,
    ),
)

ax.set_xticks(
    np.arange(
        10
    )
)

ax.set_xticklabels(
    range(
        10
    )
)

ax.set_yticks(
    np.arange(
        10
    )
)

ax.set_yticklabels(
    range(
        10
    )
)

ax.set_xlabel(
    "Predicted digit"
)

ax.set_ylabel(
    "True digit"
)

threshold = (
    selected_cm.max()
    / 2
)

for i in range(
    10
):
    for j in range(
        10
    ):
        value = int(
            selected_cm[
                i,
                j,
            ]
        )

        ax.text(
            j,
            i,
            str(
                value
            ),
            ha="center",
            va="center",
            fontsize=7.5,
            color=(
                "white"
                if value
                > threshold
                else "black"
            ),
        )

cbar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.85,
)

cbar.set_label(
    "LOPO decisions"
)

fig.tight_layout()

save_figure(
    fig,
    "Final_Model_Fig3_Selected_Confusion_Matrix",
)

plt.show()

In [19]:
# Figure 4 — selected-model F1 by physical digit.

selected_report = (
    classification_report_per_digit[
        classification_report_per_digit[
            "family"
        ]
        == SELECTED_FAMILY
    ]
    .sort_values(
        "digit"
    )
)

fig, ax = plt.subplots(
    figsize=(
        7.2,
        4.0,
    )
)

bars = ax.bar(
    selected_report[
        "digit"
    ],
    100
    * selected_report[
        "f1_score"
    ],
    width=0.62,
    color=DARK_BLUE,
    edgecolor="none",
    zorder=3,
)

for (
    bar,
    value,
) in zip(
    bars,
    100
    * selected_report[
        "f1_score"
    ],
):
    ax.annotate(
        f"{value:.1f}%",
        xy=(
            bar.get_x()
            + bar.get_width()
            / 2,
            value,
        ),
        xytext=(
            0,
            5,
        ),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=7.5,
        color=DARK_GREY,
    )

ax.set_xticks(
    range(
        10
    )
)

ax.set_xlabel(
    "True digit class"
)

ax.set_ylabel(
    "F1 score"
)

ax.set_ylim(
    0,
    105,
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda y, pos: (
            f"{y:.0f}%"
        )
    )
)

ax.grid(
    axis="y",
    alpha=0.13,
    zorder=0,
)

fig.tight_layout()

save_figure(
    fig,
    "Final_Model_Fig4_Selected_PerDigit_F1",
)

plt.show()

## 16. Freeze preparation

The selected family is retrained on all B01–B07.

For a neural family, the final epoch count is set to the **median selected epoch across all outer-fold/domain runs for that family**. This rule is fixed before the full-data retrain.

The final freeze contains six models:

- W1-CCW
- W1-CW
- W2-CCW
- W2-CW
- W4-CCW
- W4-CW

After this cell completes, B01–B07 are fully consumed as development/training data.


In [20]:
if FROZEN_DIR.exists():
    shutil.rmtree(
        FROZEN_DIR
    )

FROZEN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if SELECTED_FAMILY in NEURAL_FAMILIES:
    selected_epochs = fit_records.loc[
        fit_records[
            "family"
        ]
        == SELECTED_FAMILY,
        "selected_epoch",
    ].dropna()

    FINAL_EPOCHS = int(
        max(
            1,
            round(
                float(
                    selected_epochs.median()
                )
            ),
        )
    )
else:
    FINAL_EPOCHS = None

print(
    "Selected family:",
    SELECTED_FAMILY,
)

print(
    "Final neural epoch count:",
    FINAL_EPOCHS,
)

Selected family: Logistic Regression
Final neural epoch count: None


In [21]:
frozen_domains = []

for wheel in WHEELS:
    for direction in DIRECTIONS:
        domain_mask = (
            manifest[
                "wheel"
            ].eq(
                wheel
            )
            &
            manifest[
                "direction"
            ].eq(
                direction
            )
        )

        train_idx = manifest.index[
            domain_mask
        ].to_numpy()

        assert len(
            train_idx
        ) == 1400

        domain_seed = (
            SEED
            + 900000
            + wheel
            * 10
            + (
                1
                if direction
                == "CW"
                else 0
            )
        )

        started = time.perf_counter()

        if SELECTED_FAMILY in [
            "Logistic Regression",
            "RBF-SVM",
            "HistGradientBoosting",
        ]:
            model = ClassicalScoreModel(
                family=SELECTED_FAMILY,
                seed=domain_seed,
            )

            model.fit(
                X[
                    train_idx
                ],
                manifest.loc[
                    train_idx,
                    "y",
                ].to_numpy(
                    dtype=int
                ),
            )

            extension = ".joblib"

            filename = (
                f"W{wheel}_{direction}"
                f"{extension}"
            )

            path = (
                FROZEN_DIR
                / filename
            )

            model.save(
                path
            )

        else:
            model, scalers, final_loss_history = train_neural_fixed_epochs(
                family=SELECTED_FAMILY,
                train_idx=train_idx,
                epochs=FINAL_EPOCHS,
                seed=domain_seed,
            )

            filename = (
                f"W{wheel}_{direction}.pt"
            )

            path = (
                FROZEN_DIR
                / filename
            )

            payload = {
                "family": SELECTED_FAMILY,
                "wheel": wheel,
                "direction": direction,
                "feature_count": 695,
                "final_epochs": FINAL_EPOCHS,
                "model_config": MODEL_CONFIG,
                "state_dict": {
                    key: value.detach().cpu()
                    for key, value
                    in model.state_dict().items()
                },
            }

            if SELECTED_FAMILY == "MLP":
                payload[
                    "flat_scaler_mean"
                ] = scalers[
                    "flat"
                ].mean_

                payload[
                    "flat_scaler_scale"
                ] = scalers[
                    "flat"
                ].scale_
            else:
                payload[
                    "seq_scaler_mean"
                ] = scalers[
                    "seq"
                ].mean_

                payload[
                    "seq_scaler_scale"
                ] = scalers[
                    "seq"
                ].scale_

                payload[
                    "global_scaler_mean"
                ] = scalers[
                    "global"
                ].mean_

                payload[
                    "global_scaler_scale"
                ] = scalers[
                    "global"
                ].scale_

            torch.save(
                payload,
                path,
            )

        fit_seconds = (
            time.perf_counter()
            - started
        )

        sha256 = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

        frozen_domains.append({
            "wheel": wheel,
            "direction": direction,
            "file": filename,
            "sha256": sha256,
            "fit_seconds": fit_seconds,
            "size_bytes": path.stat().st_size,
        })

        del model

        if SELECTED_FAMILY in NEURAL_FAMILIES:
            try:
                del scalers
            except Exception:
                pass

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


freeze_manifest = {
    "model_name": (
        "Final True-Gate Model"
    ),
    "status": (
        "FROZEN_AFTER_B01_B07_DEVELOPMENT"
    ),
    "selected_family": (
        SELECTED_FAMILY
    ),
    "training_batches": (
        BATCHES
    ),
    "feature_count": 695,
    "feature_representation": (
        "Dual-full Current-only"
    ),
    "structured_deep_input": {
        "temporal": (
            "6 x 96"
        ),
        "global": 119,
    },
    "wheel_handling": (
        "wheel-specific"
    ),
    "direction_handling": (
        "separate CW/CCW"
    ),
    "repeat_training": (
        "A+B"
    ),
    "repeat_inference": (
        "within-scan score standardisation + matched A/B mean"
    ),
    "final_epochs": (
        FINAL_EPOCHS
    ),
    "domain_models": (
        frozen_domains
    ),
    "evaluation_status": (
        "Awaiting a new password collected after this freeze."
    ),
}

with open(
    FROZEN_DIR
    / "Final_TrueGate_Model_Manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        freeze_manifest,
        f,
        indent=2,
    )


archive_base = (
    RESULT_DIR
    / "Final_TrueGate_Model_FROZEN"
)

zip_path = shutil.make_archive(
    str(
        archive_base
    ),
    "zip",
    root_dir=FROZEN_DIR,
)

zip_sha256 = hashlib.sha256(
    Path(
        zip_path
    ).read_bytes()
).hexdigest()

with open(
    RESULT_DIR
    / "Final_TrueGate_Model_SHA256.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        zip_sha256
        + "\n"
    )

print(
    "Frozen package:",
    zip_path,
)

print(
    "SHA-256:",
    zip_sha256,
)

Frozen package: /content/drive/MyDrive/Padlock_Reproduction_v1/results/Final_Model_Training/Final_TrueGate_Model_FROZEN.zip
SHA-256: 104a385f55b72b0fb7b280a0e95f9b76d9304ae08015b44ef6e2a5dacebc9a4c


## 17. Save complete model-comparison report

In [22]:
candidate_scores.to_csv(
    RESULT_DIR
    / "Final_Model_Candidate_Scores.csv",
    index=False,
)

fused_candidates.to_csv(
    RESULT_DIR
    / "Final_Model_Fused_Candidate_Scores.csv",
    index=False,
)

decisions.to_csv(
    RESULT_DIR
    / "Final_Model_LOPO_Predictions.csv",
    index=False,
)

summary.to_csv(
    RESULT_DIR
    / "Final_Model_Summary.csv",
    index=False,
)

wheel_summary.to_csv(
    RESULT_DIR
    / "Final_Model_Wheel_Summary.csv",
    index=False,
)

direction_summary.to_csv(
    RESULT_DIR
    / "Final_Model_Direction_Summary.csv",
    index=False,
)

password_summary.to_csv(
    RESULT_DIR
    / "Final_Model_Password_Summary.csv",
    index=False,
)

fit_records.to_csv(
    RESULT_DIR
    / "Final_Model_Fit_Records.csv",
    index=False,
)

complexity_summary.to_csv(
    RESULT_DIR
    / "Final_Model_Complexity.csv",
    index=False,
)

classification_report_summary.to_csv(
    RESULT_DIR
    / "Final_Model_Classification_Report_Summary.csv",
    index=False,
)

classification_report_per_digit.to_csv(
    RESULT_DIR
    / "Final_Model_Classification_Report_Per_Digit.csv",
    index=False,
)

pairwise_tests.to_csv(
    RESULT_DIR
    / "Final_Model_Password_Paired_Tests.csv",
    index=False,
)

selection_table.to_csv(
    RESULT_DIR
    / "Final_Model_Selection_Table.csv",
    index=False,
)

if not neural_history.empty:
    neural_history.to_csv(
        RESULT_DIR
        / "Final_Model_Neural_Training_History.csv",
        index=False,
    )


selected_row = selection_table.iloc[
    0
]

report_lines = [
    "# Final Model Training Report",
    "",
    "## Development protocol",
    "",
    "B01-B07 were used for model development and seven-fold password-level model selection.",
    "The selected family was then retrained on all B01-B07 and frozen.",
    "A future password collected after this freeze is required for final unbiased evaluation.",
    "",
    "## Compared model families",
    "",
]

for family in MODEL_FAMILIES:
    report_lines.append(
        f"- {family}"
    )

report_lines.extend([
    "",
    "## Overall model comparison",
    "",
    "| Model | Top-1 | Top-2 | Top-3 | Macro-F1 | Weighted-F1 | Mean rank |",
    "|---|---:|---:|---:|---:|---:|---:|",
])

for row in selection_table.itertuples():
    report_lines.append(
        (
            f"| {row.family} | "
            f"{100 * row.top1:.1f}% | "
            f"{100 * row.top2:.1f}% | "
            f"{100 * row.top3:.1f}% | "
            f"{row.macro_f1:.3f} | "
            f"{row.weighted_f1:.3f} | "
            f"{row.mean_true_rank:.2f} |"
        )
    )

report_lines.extend([
    "",
    "## Selected model",
    "",
    f"**{SELECTED_FAMILY}**",
    "",
    f"Top-1: {100 * selected_row.top1:.1f}%",
    f"Top-2: {100 * selected_row.top2:.1f}%",
    f"Top-3: {100 * selected_row.top3:.1f}%",
    f"Macro-F1: {selected_row.macro_f1:.3f}",
    f"Weighted-F1: {selected_row.weighted_f1:.3f}",
    f"Mean true rank: {selected_row.mean_true_rank:.2f}",
    "",
    "## Analysis package",
    "",
    "- classification report for all model families",
    "- confusion matrix for all model families",
    "- per-digit F1",
    "- per-wheel performance",
    "- per-direction performance",
    "- per-password performance",
    "- neural train/validation loss",
    "- neural validation Top-1",
    "- early-stopping epochs",
    "- model training time and serialized size",
    "",
    "## Freeze status",
    "",
    "`Final_TrueGate_Model_FROZEN.zip` is the post-selection full B01-B07 model package.",
    "",
    "**Do not report B01-B07 replay as final unseen accuracy after this freeze.**",
])

with open(
    RESULT_DIR
    / "Final_Model_Report.md",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        "\n".join(
            report_lines
        )
        + "\n"
    )


run_info = {
    "notebook": (
        "Final_Model_Training.ipynb"
    ),
    "development_batches": (
        BATCHES
    ),
    "model_families": (
        MODEL_FAMILIES
    ),
    "outer_evaluation": (
        "7-fold leave-one-password-out"
    ),
    "neural_inner_validation": (
        "one grouped password from each outer-training set"
    ),
    "primary_metric": (
        "A/B-fused Top-1"
    ),
    "selected_family": (
        SELECTED_FAMILY
    ),
    "frozen_package": (
        "Final_TrueGate_Model_FROZEN.zip"
    ),
    "unseen_test_status": (
        "not yet performed"
    ),
}

with open(
    RESULT_DIR
    / "Final_Model_Run_Info.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        run_info,
        f,
        indent=2,
    )

print(
    "Final report saved."
)

print(
    "Selected family:",
    SELECTED_FAMILY,
)

print(
    "Next step after freeze: collect a genuinely new password."
)

Final report saved.
Selected family: Logistic Regression
Next step after freeze: collect a genuinely new password.
